# ViralCut AI — Google Colab Runner

### Quick Start:
1. Select GPU: **Runtime → Change runtime type → T4 GPU**
2. Click: **Runtime → Run all** (or run cells sequentially with ▶️)

The runner will automatically clone the repository, install dependencies, launch the server, pre-warm local Whisper AI, and generate a temporary Cloudflare Quick Tunnel URL (`trycloudflare.com`). No API keys or account setup required.

In [ ]:
%%capture
# Everything runs silently inside this setup cell

import subprocess, time, re, os, sys

# Repository URL (Single configuration point)
REPO_URL = "https://github.com/itxunknown39-web/ViralCut-AI.git"
REPO_NAME = "ViralCut-AI"

# 1) Repository clone / directory update
if not os.path.exists(REPO_NAME):
    subprocess.run(["git", "clone", REPO_URL, REPO_NAME])
os.chdir(REPO_NAME)

# 2) System dependencies & Python requirements
subprocess.run(["apt-get", "-qq", "update"])
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run(["pip", "install", "-q", "requests"])

# 3) Cloudflare Quick Tunnel binary (no token, no account)
if not os.path.exists("cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared",
    ])
    subprocess.run(["chmod", "+x", "cloudflared"])

# 4) Start existing FastAPI server
server_log = open("server.log", "w", encoding="utf-8")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT,
)

# 5) Health check loop (waits for /health to avoid Error 1033)
import requests
ready = False
for attempt in range(90):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

# 6) Start Cloudflare Quick Tunnel once server is ready
public_url = None
if ready:
    tunnel_log = open("tunnel.log", "w", encoding="utf-8")
    tunnel_proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=tunnel_log, stderr=subprocess.STDOUT,
    )
    for attempt in range(30):
        time.sleep(2)
        if os.path.exists("tunnel.log"):
            with open("tunnel.log", encoding="utf-8") as f:
                text = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
            if match:
                public_url = match.group(0)
                break

In [ ]:
# Display public URL result, System Status Grid, and Live Log Streamer
from IPython.display import HTML, display
import time, os, re, sys, subprocess

if public_url:
    display(HTML(f'''
    <div style="font-family: system-ui, sans-serif; background: #0a0b10; border: 1.5px solid #00C9FF; border-radius: 12px; padding: 20px; color: #fff; max-width: 620px; margin: 10px 0;">
        <h3 style="margin-top:0; color:#00C9FF; font-size: 20px;">🚀 ViralCut AI — by Kamran AI</h3>
        <p style="color:#9aa1b2; font-size:14px; margin-bottom: 16px;">Your AI video clipping toolkit is running live in Google Colab.</p>
        <a href="{public_url}" target="_blank" style="display: inline-block; background: #00C9FF; color: #031422; font-weight: bold; font-size: 15px; padding: 12px 24px; border-radius: 8px; text-decoration: none;">🚀 Open Dashboard</a>
        <p style="color:#6b7280; font-size:12px; margin-top: 12px; margin-bottom: 0;">Public URL: <code style="color:#00C9FF;">{public_url}</code></p>
    </div>
    '''))

    # Determine GPU status
    has_gpu = False
    gpu_name = "CPU Only"
    try:
        import torch
        if torch.cuda.is_available():
            has_gpu = True
            gpu_name = f"NVIDIA {torch.cuda.get_device_name(0)}"
    except Exception:
        pass

    # Print System Status Grid
    gpu_status = f"🟢 GPU / CUDA        : {gpu_name}" if has_gpu else "🟡 GPU / CUDA        : CPU Fallback Mode"
    print("=" * 65)
    print("📊 VIRALCUT AI — SYSTEM STATUS")
    print("=" * 65)
    print("🟢 Repository        : ViralCut-AI (main branch)")
    print("🟢 Dependencies      : Python requirements & yt-dlp ready")
    print(gpu_status)
    print("🟢 FFmpeg            : System binary installed & ready")
    print("🟢 Frontend          : 7-Screen React Dashboard synced")
    print("🟢 FastAPI           : Running on http://127.0.0.1:8000")
    print("🟢 Health Check      : Passed (200 OK)")
    print(f"🟢 Cloudflare Tunnel : Active ({public_url})")
    print("=" * 65)
    print("📡 Streaming live application logs below (Ctrl+C or Stop Cell to end):\n")

    # Live Log Streamer from server.log
    log_file = "server.log"
    seen_lines = set()
    pos = 0

    try:
        while True:
            if os.path.exists(log_file):
                with open(log_file, "r", encoding="utf-8", errors="replace") as f:
                    f.seek(pos)
                    lines = f.readlines()
                    pos = f.tell()
                    for line in lines:
                        cleaned = line.strip()
                        if not cleaned or cleaned in seen_lines:
                            continue
                        seen_lines.add(cleaned)

                        # Format & highlight logs
                        if "ERROR" in cleaned or "Exception" in cleaned or "Traceback" in cleaned:
                            print(f"\033[91m🔴 {cleaned}\033[0m")
                        elif "WARNING" in cleaned or "WARN" in cleaned:
                            print(f"\033[93m🟡 {cleaned}\033[0m")
                        elif any(k in cleaned for k in ["download", "transcribe", "select", "render", "clip", "POST /api", "GET /api"]):
                            print(f"\033[96m⚡ {cleaned}\033[0m")
                        else:
                            print(f"  {cleaned}")

            time.sleep(2)
    except KeyboardInterrupt:
        print("\n\n⏹️ Log streaming paused. FastAPI server remains active.")
elif not ready:
    print("❌ Server failed to start. Last 40 lines of server.log:\n")
    !tail -n 40 server.log
else:
    print("⚠️ Tunnel link pending. Last 40 lines of tunnel.log:\n")
    !tail -n 40 tunnel.log

---
💡 **Note**: The public URL is temporary and will change whenever the Colab session or tunnel restarts.